
# Private Market Data — Python Technical Interview Workbook

This notebook is designed for a **senior / VP-level Python technical interview** for a data-heavy financial engineering role.

It focuses on the topics most likely to appear when the role combines:

- Python
- SQL / data processing
- production engineering
- data quality
- private-market or financial datasets
- AI / ML
- testing
- scalable pipelines

The notebook is deliberately written so that a **beginner can follow the reasoning**, but each section also includes the sort of follow-up questions expected from a senior candidate.

---

## How to use this notebook

For each section:

1. Read the interview question.
2. Try to solve it yourself before running the solution.
3. Run the code.
4. Read the explanation.
5. Review the **senior-level follow-up questions**.
6. Try the small exercise.

The most important interview habit is not memorising syntax. It is being able to explain:

- why the code is correct,
- its time and space complexity,
- edge cases,
- how it behaves on large datasets,
- how you would productionise it,
- and what data-quality controls you would add.



# 1. Core Python collections: `dict`, `set`, `defaultdict`, `Counter`

These are among the most important Python tools for coding interviews.

## Why this matters

Financial and private-market datasets often need:

- grouping by fund,
- counting records,
- deduplication,
- building lookup tables,
- accumulating metrics,
- joining observations by key.

For many such problems, the best first solution uses a hash table, which in Python usually means a `dict` or `set`.



## Question 1 — Aggregate cashflows by fund

You receive cashflow records:

- capital calls
- distributions

Return total amounts by fund.

### Example input

```python
[
    {"fund": "A", "type": "call", "amount": 100},
    {"fund": "A", "type": "distribution", "amount": 30},
    {"fund": "B", "type": "call", "amount": 200},
    {"fund": "A", "type": "distribution", "amount": 20},
]
```

### Expected idea

Use a dictionary keyed by fund.

This is a common pattern:

```text
record -> key -> accumulator
```


In [ ]:

from collections import defaultdict

cashflows = [
    {"fund": "A", "type": "call", "amount": 100},
    {"fund": "A", "type": "distribution", "amount": 30},
    {"fund": "B", "type": "call", "amount": 200},
    {"fund": "A", "type": "distribution", "amount": 20},
]

def aggregate_cashflows(cashflows):
    totals = defaultdict(lambda: {"call": 0, "distribution": 0})

    for record in cashflows:
        fund = record["fund"]
        cashflow_type = record["type"]
        amount = record["amount"]

        totals[fund][cashflow_type] += amount

    return dict(totals)

aggregate_cashflows(cashflows)



### Explanation

For every record:

1. identify the fund,
2. identify the cashflow type,
3. add the amount into that fund's accumulator.

`defaultdict` automatically creates a default structure when a fund is seen for the first time.

### Complexity

If there are `n` cashflows:

- **Time:** `O(n)`
- **Space:** `O(k)` where `k` is the number of distinct funds.

### Senior-level follow-up

An interviewer may ask:

> What if there are 500 million records?

The answer should change.

You might discuss:

- reading in chunks,
- aggregating in SQL,
- distributed processing such as Spark,
- incremental aggregation,
- partitioning by fund,
- stream processing.

The important point is:

> A correct in-memory Python solution is not automatically a production-scale solution.



# 2. Deduplication and composite keys

Duplicate records are extremely common in financial-data systems.

A duplicate may mean:

- the entire record is identical,
- the same business event arrived twice,
- the same source file was resent,
- two observations have the same business key but different values.

That distinction matters.



## Question 2 — Remove duplicates while preserving order

Input:

```python
records = [
    ("fund1", "2026-06-30", 100),
    ("fund1", "2026-06-30", 100),
    ("fund2", "2026-06-30", 200),
]
```

Return unique records but preserve input order.


In [ ]:

records = [
    ("fund1", "2026-06-30", 100),
    ("fund1", "2026-06-30", 100),
    ("fund2", "2026-06-30", 200),
]

def deduplicate_preserve_order(records):
    seen = set()
    result = []

    for record in records:
        if record not in seen:
            seen.add(record)
            result.append(record)

    return result

deduplicate_preserve_order(records)



### Why use a `set`?

A set gives average `O(1)` membership checks.

Without a set, repeatedly checking whether something is already in a list could make the solution `O(n²)`.

### Private-market use case

Suppose the real business key is:

```text
fund_id + valuation_date + source
```

Then your deduplication key should reflect that.

The interviewer may then ask:

> What if two rows have the same key but different NAV values?

This is no longer simple deduplication.

You need a rule such as:

- latest `received_at` wins,
- highest-priority source wins,
- manual override wins,
- highest data-quality score wins,
- or flag the conflict for review.

That is a strong senior-level observation.


In [ ]:

observations = [
    {
        "fund_id": "A",
        "valuation_date": "2026-06-30",
        "source": "GP",
        "received_at": "2026-07-10",
        "nav": 100
    },
    {
        "fund_id": "A",
        "valuation_date": "2026-06-30",
        "source": "GP",
        "received_at": "2026-07-15",
        "nav": 105
    }
]

def latest_by_business_key(records):
    latest = {}

    for r in records:
        key = (r["fund_id"], r["valuation_date"], r["source"])

        if key not in latest or r["received_at"] > latest[key]["received_at"]:
            latest[key] = r

    return list(latest.values())

latest_by_business_key(observations)



# 3. Sorting and Top-K problems

A common interview pattern is:

> Find the largest / smallest / most frequent `k` items.

The obvious solution is sorting.

But when `k` is small relative to `n`, a heap can be more efficient.



## Question 3 — Return the top 3 funds by return


In [ ]:

returns = {
    "Fund A": 0.12,
    "Fund B": 0.25,
    "Fund C": 0.08,
    "Fund D": 0.31,
    "Fund E": 0.18,
}

top3_sorted = sorted(
    returns.items(),
    key=lambda item: item[1],
    reverse=True
)[:3]

top3_sorted



Sorting costs approximately:

```text
O(n log n)
```

If we only need a small number `k`, use `heapq.nlargest`.


In [ ]:

import heapq

top3_heap = heapq.nlargest(
    3,
    returns.items(),
    key=lambda item: item[1]
)

top3_heap



### Complexity

For small `k`:

```text
O(n log k)
```

This can be preferable to sorting the entire dataset.

### Typical follow-up

> When would you still use sorting?

When:

- the dataset is small,
- you need the complete ordering,
- clarity matters more than micro-optimisation,
- or the underlying database can sort efficiently for you.



# 4. Sliding window

Sliding-window problems appear frequently in technical interviews.

They are especially relevant to time-series data.

Typical use cases:

- rolling sums,
- rolling averages,
- detecting local changes,
- rolling volatility,
- recent-period anomalies.



## Question 4 — Maximum sum of any `k` consecutive values

A naive solution recomputes every window.

That costs roughly `O(n*k)`.

A sliding window reuses the previous calculation and reduces it to `O(n)`.


In [ ]:

def max_window_sum(values, k):
    if k <= 0:
        raise ValueError("k must be positive")
    if len(values) < k:
        raise ValueError("k cannot exceed number of values")

    current = sum(values[:k])
    best = current

    for i in range(k, len(values)):
        current += values[i]
        current -= values[i - k]
        best = max(best, current)

    return best

values = [10, 12, 9, 18, 20, 17]
max_window_sum(values, 3)



### Private-market interpretation

Imagine these are quarterly changes in NAV.

You might use the same pattern to calculate:

- rolling 4-quarter return,
- rolling change,
- maximum recent deterioration,
- rolling exposure.

The important interview insight is recognising the reusable **window pattern**, not memorising one particular problem.



# 5. Queue and BFS using `deque`

Breadth-first search is a common interview topic.

A queue processes items in:

```text
First In, First Out
```

Python's preferred queue structure for BFS is:

```python
collections.deque
```



## Question 5 — Traverse a graph with BFS

Imagine dependencies between datasets:

```text
raw_observation
    -> fund_metric
    -> peer_group
    -> benchmark
```

If an upstream observation changes, you may need to find all downstream dependencies.


In [ ]:

from collections import deque

graph = {
    "raw_observation": ["fund_metric"],
    "fund_metric": ["peer_group", "client_report"],
    "peer_group": ["benchmark"],
    "benchmark": ["api_dataset"],
    "client_report": [],
    "api_dataset": [],
}

def bfs(graph, start):
    queue = deque([start])
    visited = set([start])
    order = []

    while queue:
        node = queue.popleft()
        order.append(node)

        for neighbour in graph.get(node, []):
            if neighbour not in visited:
                visited.add(neighbour)
                queue.append(neighbour)

    return order

bfs(graph, "raw_observation")



### Why `deque`?

Removing the first element from a normal list:

```python
my_list.pop(0)
```

is inefficient because remaining elements need to shift.

`deque.popleft()` is designed for efficient queue behaviour.

### Complexity

For a graph:

```text
O(V + E)
```

where:

- `V` = vertices
- `E` = edges



# 6. Generators and processing large files

This is a particularly important topic for data-engineering roles.

A generator produces data **lazily**.

Instead of loading everything into memory, it yields one item at a time.



## Question 6 — Why is this risky?

```python
records = [parse(line) for line in open("huge_file.csv")]
```

If the file contains millions of records, the whole result may be held in memory.

A generator-style approach is safer.


In [ ]:

def number_generator(n):
    for i in range(n):
        yield i

g = number_generator(5)

print(next(g))
print(next(g))
print(list(g))



### Financial-data use case

Suppose you receive a 50 GB cashflow file.

Possible approaches:

- line-by-line generator,
- `pandas.read_csv(..., chunksize=...)`,
- PyArrow streaming,
- database bulk load,
- Spark for distributed processing.

### Interview follow-up

> What is the difference between an iterable, iterator, and generator?

- **Iterable**: something that can produce an iterator.
- **Iterator**: object with `__next__()`.
- **Generator**: an iterator created by a function using `yield`.

This topic often distinguishes someone who merely uses Python from someone who understands it.



# 7. Common Python pitfalls

Senior Python interviews often include short conceptual questions.

These are easy marks if you know them well.



## Mutable default arguments


In [ ]:

def bad_add(item, items=[]):
    items.append(item)
    return items

print(bad_add("A"))
print(bad_add("B"))  # unexpected persistence



The default list is created once, not on every function call.

Correct version:


In [ ]:

def good_add(item, items=None):
    if items is None:
        items = []

    items.append(item)
    return items

print(good_add("A"))
print(good_add("B"))



## `is` vs `==`

- `==` checks equality.
- `is` checks object identity.

Use:

```python
if value is None:
    ...
```

not:

```python
if value == None:
    ...
```



# 8. Dataclasses and domain modelling

A senior engineer should be able to move beyond unstructured dictionaries.

`dataclass` is useful for expressing domain objects clearly.


In [ ]:

from dataclasses import dataclass
from datetime import date

@dataclass(frozen=True)
class FundObservation:
    fund_id: str
    valuation_date: date
    nav: float
    currency: str

obs = FundObservation(
    fund_id="F123",
    valuation_date=date(2026, 6, 30),
    nav=150_000_000,
    currency="GBP",
)

obs



### Why `frozen=True`?

It makes the dataclass immutable after creation.

For financial-data pipelines this can be useful because:

- accidental mutation becomes harder,
- objects behave more predictably,
- immutable records are easier to reason about,
- auditability improves.

An interviewer may ask whether immutability is always desirable.

Correct answer: **no**.

Use it where it improves the domain model; do not force it everywhere.



# 9. OOP design: extensible validation framework

This is a realistic senior interview question.

> Design a validation framework where new rules can be added without constantly changing the core engine.

This lets the interviewer test:

- abstractions,
- composition,
- interfaces,
- maintainability,
- testability.


In [ ]:

from abc import ABC, abstractmethod

class ValidationRule(ABC):

    @abstractmethod
    def validate(self, record):
        """Return None if valid, otherwise return an error message."""
        raise NotImplementedError


class PositiveNAVRule(ValidationRule):

    def validate(self, record):
        nav = record.get("nav")

        if nav is None:
            return "NAV is missing"

        if nav < 0:
            return "NAV cannot be negative"

        return None


class CurrencyRule(ValidationRule):

    def __init__(self, allowed):
        self.allowed = set(allowed)

    def validate(self, record):
        if record.get("currency") not in self.allowed:
            return "Invalid currency"

        return None


class Validator:

    def __init__(self, rules):
        self.rules = rules

    def validate(self, record):
        errors = []

        for rule in self.rules:
            error = rule.validate(record)

            if error is not None:
                errors.append(error)

        return errors


validator = Validator([
    PositiveNAVRule(),
    CurrencyRule({"GBP", "USD", "EUR"}),
])

validator.validate({"fund": "A", "nav": -10, "currency": "XXX"})



### Senior-level discussion

Why is this better than one giant `if/elif` function?

Because rules are:

- independently testable,
- composable,
- easier to add,
- easier to disable,
- easier to version.

A production system might also include:

```text
rule_id
rule_version
severity
owner
description
effective_from
effective_to
```

That lets you audit exactly which rule failed for a particular data version.



# 10. Data quality controls

For a private-market data platform, data quality is not optional.

A useful framework separates checks into layers.

## 1. Schema quality

Examples:

- required fields present,
- correct types,
- valid date format,
- valid currency code.

## 2. Business-rule quality

Examples:

- NAV must not be negative,
- paid-in capital should not exceed committed capital without explanation,
- valuation date should be plausible.

## 3. Referential quality

Examples:

- every fund references a valid manager,
- every portfolio company ID exists in the entity master.

## 4. Temporal quality

Examples:

- records should not move backwards unexpectedly,
- stale data should be identified,
- future-dated records should be investigated.

## 5. Statistical quality

Examples:

- sudden 500% NAV jump,
- unusual missing-value rate,
- unexpected changes in record count,
- distribution shift.


In [ ]:

VALID_CURRENCIES = {"GBP", "USD", "EUR"}

def validate_record(record):
    errors = []

    if not record.get("fund_id"):
        errors.append("missing fund_id")

    nav = record.get("nav")

    if nav is None:
        errors.append("missing nav")
    elif nav < 0:
        errors.append("negative nav")

    if record.get("currency") not in VALID_CURRENCIES:
        errors.append("invalid currency")

    return errors


sample_records = [
    {"fund_id": "A", "nav": 100, "currency": "GBP"},
    {"fund_id": "B", "nav": -10, "currency": "USD"},
    {"fund_id": None, "nav": 20, "currency": "XXX"},
]

[(r, validate_record(r)) for r in sample_records]



### Interview follow-up

> Should a bad record crash the whole batch?

Usually not.

A mature architecture often separates:

```text
valid data -> continue processing

invalid data -> quarantine + reason + alert
```

You may also classify errors:

```text
ERROR
WARNING
INFO
```

A negative NAV might be an error.

A large quarter-on-quarter movement might be a warning that requires review.



# 11. Statistical anomaly detection

This is particularly relevant when a role combines software engineering with quantitative / ML workflows.

The interviewer may ask:

> How would you detect suspicious values in a financial dataset?

A strong answer starts simple.

Do not immediately jump to deep learning.



## 11.1 Z-score

For observation \(x\):

\[
z = \frac{x - \mu}{\sigma}
\]

where:

- \(\mu\) = mean
- \(\sigma\) = standard deviation

A common rule is:

```text
|z| > 3
```

But this works best when the data is reasonably well behaved and not heavily contaminated by outliers.


In [ ]:

import numpy as np

values = np.array([100, 102, 98, 101, 99, 103, 500], dtype=float)

mean = values.mean()
std = values.std(ddof=1)

z_scores = (values - mean) / std

list(zip(values, z_scores))



Notice the problem:

The extreme observation `500` also changes the mean and standard deviation.

That means the outlier influences the baseline used to detect itself.

For dirty financial data, robust statistics are often better.



## 11.2 Robust Z-score with Median Absolute Deviation

Median:

\[
m = median(x)
\]

Median absolute deviation:

\[
MAD = median(|x_i - m|)
\]

Robust score:

\[
z_r = 0.6745 \times \frac{x_i - m}{MAD}
\]

The median and MAD are much less sensitive to extreme values.


In [ ]:

def robust_z_scores(values):
    x = np.asarray(values, dtype=float)

    median = np.median(x)
    mad = np.median(np.abs(x - median))

    if mad == 0:
        return np.zeros_like(x)

    return 0.6745 * (x - median) / mad


scores = robust_z_scores(values)

list(zip(values, scores))


In [ ]:

anomaly_indices = np.where(np.abs(scores) > 3.5)[0]
anomaly_indices



### Why this is useful in private markets

Potential anomaly examples:

- NAV jumps 700% quarter-on-quarter,
- commitment amount changes by a factor of 10,
- reported distribution is far outside historical behaviour,
- fund size differs sharply from neighbouring observations.

However:

> Statistical anomaly does not mean business error.

A real event can be statistically unusual.

The correct architecture is often:

```text
statistical detector
        ↓
risk score / explanation
        ↓
business validation
        ↓
human review where needed
```



# 12. Rolling anomaly detection for time series

For time-series data, comparing a point to the whole historical distribution can be misleading.

Instead, compare it to a recent rolling window.


In [ ]:

import pandas as pd

series = pd.Series([
    100, 101, 102, 100, 99, 101, 102, 250, 103, 104
])

rolling_mean = series.rolling(window=4).mean().shift(1)
rolling_std = series.rolling(window=4).std().shift(1)

rolling_z = (series - rolling_mean) / rolling_std

pd.DataFrame({
    "value": series,
    "rolling_mean": rolling_mean,
    "rolling_std": rolling_std,
    "rolling_z": rolling_z,
})



### Why `.shift(1)`?

Without shifting, the current value would contribute to the baseline used to judge itself.

Using only prior observations avoids that form of leakage.

This idea is extremely important in:

- forecasting,
- anomaly detection,
- model backtesting,
- point-in-time financial datasets.



# 13. Pandas: the operations you should know without an IDE

For a data-heavy interview, be comfortable with:

```python
groupby
merge
sort_values
drop_duplicates
isna
fillna
shift
diff
pct_change
rolling
transform
pivot_table
```



## Question 7 — Calculate percentage NAV change per fund


In [ ]:

df = pd.DataFrame({
    "fund": ["A", "A", "B", "B"],
    "date": pd.to_datetime([
        "2026-03-31",
        "2026-06-30",
        "2026-03-31",
        "2026-06-30"
    ]),
    "nav": [100, 120, 200, 190]
})

df = df.sort_values(["fund", "date"])

df["nav_change_pct"] = (
    df.groupby("fund")["nav"]
      .pct_change()
)

df



### Explanation

`groupby("fund")` separates the data by fund.

`pct_change()` compares each value with the previous value within that fund.

For Fund A:

```text
100 -> 120 = +20%
```

For Fund B:

```text
200 -> 190 = -5%
```



# 14. Pandas joins and cardinality

Joins are a common source of silent data-quality problems.

Suppose funds reference managers.


In [ ]:

funds = pd.DataFrame({
    "fund_id": [1, 2, 3],
    "manager_id": [10, 20, 20]
})

managers = pd.DataFrame({
    "manager_id": [10, 20],
    "manager_name": ["ABC Capital", "XYZ Partners"]
})

merged = funds.merge(
    managers,
    on="manager_id",
    how="left",
    validate="many_to_one"
)

merged



### Why `validate="many_to_one"` is valuable

It asserts that:

```text
many funds may map to one manager
```

but the manager table should contain only one row per manager ID.

If the manager table unexpectedly contains duplicates, Pandas will raise an error instead of silently multiplying rows.

This is a very strong data-engineering habit.



# 15. Latest valid observation by fund

This is one of the most realistic private-market coding questions.

You receive multiple observations.

Some are corrections.

Return the latest observation for each fund.


In [ ]:

from datetime import datetime

records = [
    {
        "fund": "A",
        "effective_date": "2026-06-30",
        "received_at": "2026-07-20",
        "nav": 100
    },
    {
        "fund": "A",
        "effective_date": "2026-06-30",
        "received_at": "2026-08-01",
        "nav": 105
    },
    {
        "fund": "B",
        "effective_date": "2026-06-30",
        "received_at": "2026-07-22",
        "nav": 200
    }
]

def latest_received_by_fund(records):
    latest = {}

    for record in records:
        fund = record["fund"]
        received = datetime.fromisoformat(record["received_at"])

        if fund not in latest:
            latest[fund] = (received, record)
        elif received > latest[fund][0]:
            latest[fund] = (received, record)

    return {
        fund: record
        for fund, (_, record) in latest.items()
    }

latest_received_by_fund(records)



### Important senior-level distinction

There are two different dates:

```text
effective_date
received_at
```

These mean different things.

A NAV may be:

```text
effective on 30 June
received on 1 August
```

If a correction arrives on 15 September, overwriting the old row destroys history.

A better production design preserves observations and records which one is currently trusted.

This leads naturally into **bitemporal data modelling**.



# 16. Point-in-time data and leakage

This topic is extremely important for financial ML.

Suppose you want to train a model to predict something as of:

```text
30 June
```

A record whose economic date is 30 June but which was only received on 10 August **must not** be available to the June prediction.

Otherwise you have data leakage.

The training dataset must answer:

> What did the system actually know at prediction time?


In [ ]:

point_in_time = pd.Timestamp("2026-06-30")

observations = pd.DataFrame({
    "fund": ["A", "A", "B"],
    "effective_date": pd.to_datetime([
        "2026-03-31",
        "2026-06-30",
        "2026-03-31"
    ]),
    "received_at": pd.to_datetime([
        "2026-04-20",
        "2026-08-10",
        "2026-04-25"
    ]),
    "nav": [100, 120, 200]
})

available_at_prediction_time = observations[
    observations["received_at"] <= point_in_time
]

available_at_prediction_time



This is the kind of subtle issue that can completely invalidate a machine-learning backtest.

Mentioning point-in-time correctness is a strong interview answer.



# 17. Testing with `pytest`

The role of testing in data systems is broader than ordinary unit testing.

You may need:

- unit tests,
- integration tests,
- schema tests,
- data-quality tests,
- golden-dataset regression tests,
- contract tests,
- property-based tests.



## Question 8 — Test a financial metric


In [ ]:

def calculate_tvpi(distributions, nav, paid_in):
    if paid_in <= 0:
        raise ValueError("paid_in must be positive")

    return (distributions + nav) / paid_in


print(calculate_tvpi(
    distributions=40,
    nav=80,
    paid_in=100
))



A normal `pytest` test would look like:

```python
def test_tvpi():
    assert calculate_tvpi(40, 80, 100) == 1.2

def test_invalid_paid_in():
    with pytest.raises(ValueError):
        calculate_tvpi(40, 80, 0)
```

### Golden dataset test

For a financial calculation engine, it can be useful to keep a known dataset where expected outputs have already been independently validated.

Then every code change runs against that dataset.

This detects accidental changes to:

- IRR,
- TVPI,
- DPI,
- benchmark calculations,
- ranking logic.



# 18. Exceptions and quarantine patterns

A production data pipeline should distinguish:

```text
unexpected system failure

from

expected bad-data condition
```


In [ ]:

class DataValidationError(Exception):
    pass


def validated_nav(nav):
    if nav is None:
        raise DataValidationError("NAV is missing")

    if nav < 0:
        raise DataValidationError(f"NAV cannot be negative: {nav}")

    return nav


try:
    validated_nav(-10)
except DataValidationError as exc:
    print("Quarantine record:", exc)



### Why this matters

You often do **not** want one invalid record to crash a 10-million-row batch.

Instead:

```text
good rows -> continue

bad rows -> quarantine

quarantine -> reason + source + timestamp + rule version
```

This gives both resilience and auditability.



# 19. Concurrency: threads, processes and async

This is a common conceptual interview topic.

## Threads

Usually useful for **I/O-bound** work:

- reading files,
- API calls,
- database calls.

## Multiprocessing

Useful for **CPU-bound Python** workloads.

## Asyncio

Useful when one process must manage a very large number of concurrent I/O operations.

Examples:

- calling hundreds of APIs,
- waiting on many network operations.

## GIL

In standard CPython, the Global Interpreter Lock prevents multiple threads from executing Python bytecode simultaneously in one process.

That means:

- threads are often good for I/O,
- they usually do not provide true CPU parallelism for pure-Python computation.

For CPU-heavy work, options include:

- multiprocessing,
- NumPy/native code,
- Spark,
- distributed workers.



# 20. ML anomaly detection with Isolation Forest

Statistical methods are a good baseline.

When relationships become multidimensional, an ML model may help.

Isolation Forest is a common unsupervised anomaly detector.


In [1]:

from sklearn.ensemble import IsolationForest
import pandas as pd

X = pd.DataFrame({
    "nav_change_pct": [0.02, 0.01, -0.03, 0.04, 0.02, 5.50],
    "cashflow_ratio": [0.10, 0.08, 0.12, 0.09, 0.11, 4.20],
})

model = IsolationForest(
    contamination=0.15,
    random_state=42
)

predictions = model.fit_predict(X)

result = X.copy()
result["prediction"] = predictions
result["is_anomaly"] = result["prediction"] == -1

result


,nav_change_pct,cashflow_ratio,prediction,is_anomaly
0,0.02,0.10,1,False
1,0.01,0.08,1,False
2,-0.03,0.12,1,False
3,0.04,0.09,1,False
4,0.02,0.11,1,False
5,5.50,4.20,-1,True



### What does `contamination` mean?

It expresses the expected proportion of anomalies used to determine the decision threshold.

### Important interview point

Do not treat the model prediction as truth.

A practical anomaly pipeline often produces:

```text
anomaly score
reason / features
source information
comparison with history
review decision
```

### Evaluation when labels are sparse

Possible techniques:

- precision among top alerts,
- analyst acceptance rate,
- historical known incidents,
- manually labelled samples,
- synthetic anomaly injection,
- stability across periods,
- downstream business impact.



# 21. ML model-development lifecycle

For a senior AI/ML-enabled engineering role, be ready to explain the full lifecycle.

```text
Problem definition
       ↓
Point-in-time dataset
       ↓
Feature engineering
       ↓
Train / validation / test split
       ↓
Baseline model
       ↓
Experiment tracking
       ↓
Evaluation
       ↓
Model approval
       ↓
Model registry
       ↓
Deployment
       ↓
Monitoring
       ↓
Retraining / rollback
```

## What should be versioned?

A reproducible experiment should identify:

```text
code version
data version
feature version
parameters
environment
model artifact
metrics
```



# 22. DVC, experiment tracking and model registry

These solve related but different problems.

## Git

Versions:

- source code,
- configuration,
- lightweight metadata.

## DVC

Useful for:

- large dataset versioning,
- dataset lineage,
- linking data versions to Git commits,
- reproducible pipelines.

## MLflow-style experiment tracking

Tracks:

- parameters,
- metrics,
- artifacts,
- run IDs,
- experiment comparisons.

## Model registry

Tracks:

- model versions,
- lifecycle stage,
- approvals,
- deployment status.

Do not conflate these.

A strong interview answer sounds like:

> Git versions the code. DVC can version the data dependencies. Experiment tracking records what happened during each training run. The model registry governs which trained artifact is approved and deployed.



# 23. SQL patterns you may be asked alongside Python

Even if the interview is labelled Python, a data-heavy role may mix in SQL.

Know:

```text
JOIN
GROUP BY
HAVING
CTE
ROW_NUMBER
RANK
LAG
LEAD
SUM OVER
```

A very common example is calculating change from the previous observation.



```sql
SELECT
    fund_id,
    valuation_date,
    nav,
    nav - LAG(nav) OVER (
        PARTITION BY fund_id
        ORDER BY valuation_date
    ) AS nav_change
FROM fund_nav;
```

Pandas equivalent:


In [ ]:

df2 = pd.DataFrame({
    "fund_id": ["A", "A", "A", "B", "B"],
    "valuation_date": pd.to_datetime([
        "2026-03-31",
        "2026-06-30",
        "2026-09-30",
        "2026-03-31",
        "2026-06-30",
    ]),
    "nav": [100, 120, 115, 200, 190],
})


In [ ]:

df2 = df2.sort_values(["fund_id", "valuation_date"])

df2["nav_change"] = (
    df2.groupby("fund_id")["nav"]
       .diff()
)

df2


,fund_id,valuation_date,nav,nav_change
0,A,2026-03-31,100,NaN
1,A,2026-06-30,120,20.0
2,A,2026-09-30,115,-5.0
3,B,2026-03-31,200,NaN
4,B,2026-06-30,190,-10.0



# 24. Code-review style question

At senior level, you may be asked to critique code rather than only write code.

Consider:


In [ ]:

def bad_process(data):
    result = []

    for x in data:
        if x["nav"]:
            if x["nav"] > 0:
                result.append(x)

    return result



## Problems

Possible issues:

1. `x["nav"]` may raise `KeyError`.
2. `0` is treated as false even though it may be a meaningful value.
3. `None` handling is unclear.
4. No explanation of rejected records.
5. No schema contract.
6. Poor naming.
7. No typing.
8. No observability.
9. No tests.

A cleaner basic version:


In [ ]:

from typing import Iterable

def valid_nav_records(records: Iterable[dict]) -> list[dict]:
    result = []

    for record in records:
        nav = record.get("nav")

        if nav is None:
            continue

        if nav < 0:
            continue

        result.append(record)

    return result



For a production system, you would likely go further and return:

```text
accepted records
rejected records
rejection reasons
```

instead of silently dropping data.



# 25. Realistic mock question — end-to-end

## Scenario

Your platform ingests quarterly NAV values from several external sources.

You must:

1. validate the records,
2. remove exact duplicates,
3. choose the latest received observation for each `(fund, effective_date, source)`,
4. flag unusually large changes,
5. return both accepted and quarantined records.

Think through the problem before reading the solution below.


In [ ]:

from collections import defaultdict
from datetime import datetime

records = [
    {
        "fund": "A",
        "effective_date": "2026-03-31",
        "received_at": "2026-04-20",
        "source": "GP",
        "nav": 100.0,
        "currency": "GBP"
    },
    {
        "fund": "A",
        "effective_date": "2026-06-30",
        "received_at": "2026-07-20",
        "source": "GP",
        "nav": 110.0,
        "currency": "GBP"
    },
    {
        "fund": "A",
        "effective_date": "2026-06-30",
        "received_at": "2026-07-25",
        "source": "GP",
        "nav": 111.0,
        "currency": "GBP"
    },
    {
        "fund": "B",
        "effective_date": "2026-06-30",
        "received_at": "2026-07-21",
        "source": "LP",
        "nav": -50.0,
        "currency": "USD"
    }
]


In [ ]:

def process_observations(records):
    valid_currencies = {"GBP", "USD", "EUR"}

    quarantined = []
    valid = []

    # Step 1: validation
    for r in records:
        errors = []

        if not r.get("fund"):
            errors.append("missing fund")

        if r.get("nav") is None:
            errors.append("missing nav")
        elif r["nav"] < 0:
            errors.append("negative nav")

        if r.get("currency") not in valid_currencies:
            errors.append("invalid currency")

        if errors:
            quarantined.append({
                "record": r,
                "errors": errors
            })
        else:
            valid.append(r)

    # Step 2: deduplicate / keep latest version by business key
    latest = {}

    for r in valid:
        key = (
            r["fund"],
            r["effective_date"],
            r["source"]
        )

        received_at = datetime.fromisoformat(r["received_at"])

        if key not in latest:
            latest[key] = (received_at, r)
        elif received_at > latest[key][0]:
            latest[key] = (received_at, r)

    accepted = [
        r for _, r in latest.values()
    ]

    # Step 3: sort chronologically
    accepted.sort(
        key=lambda r: (
            r["fund"],
            r["effective_date"]
        )
    )

    # Step 4: simple change-based flag
    previous_nav = {}
    reviewed = []

    for r in accepted:
        r = dict(r)
        fund = r["fund"]

        if fund in previous_nav:
            change = (
                r["nav"] - previous_nav[fund]
            ) / previous_nav[fund]

            r["nav_change_pct"] = change
            r["large_change_flag"] = abs(change) > 0.50

        else:
            r["nav_change_pct"] = None
            r["large_change_flag"] = False

        previous_nav[fund] = r["nav"]
        reviewed.append(r)

    return reviewed, quarantined


accepted, quarantined = process_observations(records)

accepted, quarantined



## What the interviewer may ask next

### Why 50%?

It is only an example threshold.

In production, the threshold should come from:

- historical distributions,
- business knowledge,
- robust statistics,
- segment-specific thresholds,
- or model calibration.

### Should a large change be rejected?

Usually not automatically.

It should normally be:

```text
flagged for investigation
```

because a large movement may be genuine.

### What if there are millions of records?

Move away from one-process in-memory logic.

Potential approaches:

- SQL/window functions,
- Pandas chunking,
- Spark,
- partitioned processing,
- incremental state.

### How would you make it reproducible?

Record:

- input dataset version,
- validation-rule version,
- code version,
- configuration,
- output dataset version.



# 26. High-probability rapid-fire Python questions

You should be able to answer these clearly.

## Data structures

**Q:** Difference between list and tuple?  
**A:** Lists are mutable; tuples are immutable.

**Q:** When can a tuple be a dictionary key?  
**A:** When all of its elements are hashable.

**Q:** Average dictionary lookup complexity?  
**A:** `O(1)`.

**Q:** Why use a set?  
**A:** Fast membership checking and uniqueness.

---

## Python behaviour

**Q:** `is` vs `==`?  
**A:** Identity vs equality.

**Q:** Why avoid mutable default arguments?  
**A:** The same object is reused across calls.

**Q:** Shallow vs deep copy?  
**A:** Shallow copy reuses nested references; deep copy recursively copies nested objects.

---

## Iteration

**Q:** Why generator instead of list?  
**A:** Lazy processing and lower memory use.

**Q:** Iterator protocol?  
**A:** `__iter__()` and `__next__()`.

---

## Performance

**Q:** Threads vs processes?  
**A:** Threads for I/O-bound work; processes often better for CPU-bound pure Python.

**Q:** What is the GIL?  
**A:** A CPython lock that allows one thread at a time to execute Python bytecode in a process.

---

## Engineering

**Q:** Why use decorators?  
**A:** Cross-cutting behaviours such as logging, timing, retries, metrics, auth.

**Q:** Why context managers?  
**A:** Deterministic setup and cleanup of resources.

**Q:** Why dataclasses?  
**A:** Concise, explicit domain objects with generated boilerplate.



# 27. Practice exercises

Try these yourself.

## Exercise 1

Given transactions:

```python
[
    {"fund": "A", "amount": 10},
    {"fund": "B", "amount": 20},
    {"fund": "A", "amount": 30},
]
```

Return:

```python
{"A": 40, "B": 20}
```

---

## Exercise 2

Given a list of NAV observations, return the largest absolute quarter-on-quarter percentage move.

---

## Exercise 3

Given two Pandas tables:

```text
funds
managers
```

left-join managers to funds and enforce that the manager table is unique by `manager_id`.

---

## Exercise 4

Write a function that returns both:

```text
valid_records
invalid_records
```

where invalid records include an error reason.

---

## Exercise 5

Use a rolling 5-observation window to identify a value more than 3 standard deviations from the previous window.

---

## Exercise 6

Implement BFS using `deque`.

---

## Exercise 7

Use `heapq` to return the top 5 funds by return.

---

## Exercise 8

Write tests for:

```python
calculate_tvpi(distributions, nav, paid_in)
```

including edge cases.

---

## Exercise 9

Explain how you would process a 100 GB CSV without loading it entirely into memory.

---

## Exercise 10

Explain how you would prevent future information from leaking into a historical ML training dataset.



# 28. Final interview checklist

Before the interview, make sure you can code or explain the following without relying heavily on an IDE.

## Python

- `dict`
- `set`
- `defaultdict`
- `Counter`
- `deque`
- `heapq`
- list/dict comprehensions
- sorting with `key=`
- generators
- exceptions
- dataclasses
- classes / composition
- decorators
- context managers

## Algorithms

- arrays / strings
- hash maps
- two pointers
- sliding window
- sorting
- binary search
- stack
- queue
- BFS / DFS
- heap
- basic dynamic programming

## Pandas

- `groupby`
- `merge`
- `sort_values`
- `drop_duplicates`
- `shift`
- `diff`
- `pct_change`
- `rolling`
- missing values
- join cardinality

## Data engineering

- deduplication
- idempotency
- chunking
- large-file processing
- data validation
- quarantine
- lineage
- data versioning
- point-in-time correctness

## Testing

- unit tests
- integration tests
- schema tests
- golden datasets
- regression tests
- property-based testing

## ML

- statistical anomaly detection
- MAD / robust z-score
- rolling baselines
- Isolation Forest
- leakage
- train / validation / test
- experiment tracking
- DVC-style dataset versioning
- model registry
- drift monitoring

---

# Most important principle

A strong senior candidate should be able to move through this progression:

```text
working Python solution
        ↓
correctness
        ↓
complexity
        ↓
edge cases
        ↓
data quality
        ↓
production scale
        ↓
testing
        ↓
observability
        ↓
reproducibility
```

That progression is often more important than producing the cleverest algorithm.


## Role Upgrade Pack: Senior Quant Data Engineering Interview

> Use this section as a 75-minute simulation for roles that blend Python, data engineering, private-market analytics, and production reliability.

### Core competencies interviewers test
- Correctness under ambiguity
- Data modeling and schema rigor
- Performance at realistic scale
- Robustness (nulls, duplicates, stale snapshots, bad types)
- Explainability: trade-offs, not just code
- Production readiness: tests, monitoring, idempotency
- Business context: what the metric means for portfolio decisions

> Answer framework: **State assumptions -> present baseline solution -> analyze complexity -> improve for scale -> add data-quality checks -> discuss production rollout**.

## 75-Minute Mock Interview Blueprint

> Recommended pacing for one complete practice cycle.

1. **0-10 min:** Clarify assumptions and data contracts.
2. **10-35 min:** Solve two coding prompts (clean, readable functions).
3. **35-50 min:** Add tests for edge cases and malformed records.
4. **50-65 min:** Optimize one bottleneck and justify the trade-offs.
5. **65-75 min:** Give a production design summary (batch + incremental + observability).

### High-impact follow-up prompts
- How would this behave on 100M rows with late-arriving updates?
- What keys make this pipeline idempotent?
- Which checks block publication vs raise warnings?
- How do you reconcile SQL and Python implementations?
- What should be measured in production dashboards?

In [ ]:
from dataclasses import dataclass
from typing import List

@dataclass
class InterviewScorecard:
    correctness: int
    complexity_reasoning: int
    edge_cases: int
    communication: int
    production_mindset: int

    def total(self) -> int:
        return (
            self.correctness
            + self.complexity_reasoning
            + self.edge_cases
            + self.communication
            + self.production_mindset
        )

    def verdict(self) -> str:
        score = self.total()
        if score >= 22:
            return "Strong hire signal"
        if score >= 17:
            return "Proceed to next round"
        return "Needs another practice cycle"

def summarize_feedback(strengths: List[str], risks: List[str]) -> str:
    return (
        "Strengths: " + "; ".join(strengths) + "\n"
        "Risks to close: " + "; ".join(risks)
    )

sample = InterviewScorecard(4, 4, 3, 4, 4)
print("Score:", sample.total(), "|", sample.verdict())
print(summarize_feedback(
    strengths=["clear assumptions", "good decomposition"],
    risks=["add more property tests", "explicit null-policy"]
))